In [ ]:
import os
from IPython.display import display, update_display, Markdown
from openai import OpenAI
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
from jobsearchapi import search_job

job_title = 'AI engineer'
country = 'Singapore'
query = job_title + 'in ' + country

jobs = search_job(query)

print(jobs)

In [ ]:
#Refine extracted descriptions only
search_result = []

for each in jobs:
    if each['description']:
        search_result.append(each['title'] + 'in ' + each['company'])
        search_result.append(each['description'])

print(f'Total job found: {len(jobs) // 2}')

active_job = '\n'.join(search_result)

In [ ]:
GEMINI_BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/'
GEMINI_API_KEY = os.getenv('GOOGLE_API_KEY')
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=GEMINI_API_KEY)
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'

In [ ]:
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
OLLAMA_API_KEY = 'ollama' # can be any words, not important
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)
OLLAMA_MODEL = 'gpt-oss:latest'

In [ ]:
system_prompt = """
You are a helpful education coach to help to design a professional 4 weeks plan for the user based on the job title. 
If the job title is not valid, please say so and ask user to input the correct job title again. 
If the job title is valid, Response in markdown. Do not wrap the markdown with code block - reply only in markdown.
"""

user_prompt = f"""
You will be provided a list of extracted similar job from the list below.
List of active job available in the user country:
{active_job}
You will analyze the job title provided by user, think thoroughly before providing a professional 4 weeks plan regarding the structures of the learning path, provide useful link for the learning & tips to achieve success for the user based on the job requirements from similar job provided above.
Lastly, highlight the most frequent or top few job requirements / skills needed from most company extracted above.
This is the job title user wish to expertise on:
Job Title: {job_title}
"""

In [ ]:
def create_career_plan(model=GEMINI_MODEL, stream=False):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt}
    ]
    selected_model = None

    if model == GEMINI_MODEL:
        selected_model = gemini

    elif model == OLLAMA_MODEL:
        selected_model = ollama
    
    else:
        return f'LLM API Key for {model} is not set up. Please use other LLM product.'

    print(f'Creating a 4-week plan for {job_title} using model {model}...')
        
    if not stream:
        response = selected_model.chat.completions.create(model=model, messages=messages)
        result = response.choices[0].message.content
        display(Markdown(result))

    else:
        stream = selected_model.chat.completions.create(model=model, messages=messages, stream=True)
        response = ''
        display_handle = display(Markdown(response), display_id=True)

        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
create_career_plan(model=GEMINI_MODEL, stream=True)

In [ ]:
create_career_plan(model=GEMINI_MODEL, stream=False)

In [ ]:
create_career_plan(model=OLLAMA_MODEL, stream=True)

In [ ]:
create_career_plan(model=OLLAMA_MODEL, stream=False)